<a href="https://colab.research.google.com/github/SarwatMajeed24/PIAIC/blob/main/totp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyotp

In [ ]:
pip install qrcode Pillow


In [ ]:
import pyotp
import qrcode
from PIL import Image

def generate_secret():
    # Generates a random base32 secret. Keep this secret safe.
    return pyotp.random_base32()

def generate_totp(secret):
    totp = pyotp.TOTP(secret)
    return totp.now()

def generate_qr_code(secret, user_email, issuer_name="MyApp"):
    # Creates a provisioning URI for the QR code
    totp = pyotp.TOTP(secret)
    uri = totp.provisioning_uri(name=user_email, issuer_name=issuer_name)

    # Generate QR code
    qr = qrcode.make(uri)
    qr.save("qrcode.png")
    print("QR code saved as qrcode.png. Scan it with your authenticator app.")

if __name__ == "__main__":
    user_email = "user@example.com"  # Replace with the user's email
    secret = generate_secret()
    print(f"Secret key (keep this safe): {secret}")

    generate_qr_code(secret, user_email)

    current_totp = generate_totp(secret)
    print(f"Current OTP: {current_totp}")


Secret key (keep this safe): RLHT4TIZMQJQAG53BO6BMR6BDSAN575X
QR code saved as qrcode.png. Scan it with your authenticator app.
Current OTP: 997765


In [ ]:
import pyotp

def verify_totp(secret, user_input):
    totp = pyotp.TOTP(secret)
    return totp.verify(user_input)

if __name__ == "__main__":
     secret = input("Enter your secret key: ").strip()
     user_input = input("Enter the OTP: ").strip()

     if verify_totp(secret, user_input):
        print("OTP is valid!")
     else:
        print("Invalid OTP. Please try again.")


KeyboardInterrupt: Interrupted by user

In [ ]:
from flask import Flask, render_template, request, redirect, url_for, flash
import pyotp
import qrcode
import io
import base64

app = Flask(__name__)
app.secret_key = 'your_secret_key'

# In-memory storage for demonstration purposes
user_secrets = {}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        user_email = request.form['email']
        secret = pyotp.random_base32()
        user_secrets[user_email] = secret
        totp = pyotp.TOTP(secret)
        uri = totp.provisioning_uri(name=user_email, issuer_name="MyApp")

        # Generate QR code
        qr = qrcode.QRCode(version=1, box_size=10, border=5)
        qr.add_data(uri)
        qr.make(fit=True)
        img = qr.make_image(fill='black', back_color='white')
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

        return render_template('register_success.html', email=user_email, qr_code=img_b64, secret=secret)
    return render_template('register.html')

@app.route('/verify', methods=['GET', 'POST'])
def verify():
    if request.method == 'POST':
        user_email = request.form['email']
        otp = request.form['otp']
        secret = user_secrets.get(user_email)
        if not secret:
            flash("User not found.")
            return redirect(url_for('verify'))
        totp = pyotp.TOTP(secret)
        if totp.verify(otp):
            flash("OTP is valid!", "success")
        else:
            flash("Invalid OTP. Please try again.", "danger")
        return redirect(url_for('verify'))
    return render_template('verify.html')

if __name__ == '__main__':
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat


In [ ]:
<!doctype html>
<html lang="en">
<head>
    <title>TOTP App</title>
</head>
<body>
    <h1>Welcome to the TOTP App</h1>
    <a href="{{ url_for('register') }}">Register</a> |
    <a href="{{ url_for('verify') }}">Verify OTP</a>
</body>
</html>


In [ ]:
<!doctype html>
<html lang="en">
<head>
    <title>Register - TOTP App</title>
</head>
<body>
    <h1>Register</h1>
    <form method="post">
        <label for="email">Email:</label>
        <input type="email" name="email" required>
        <button type="submit">Register</button>
    </form>
    <a href="{{ url_for('index') }}">Home</a>
</body>
</html>


In [ ]:
<!doctype html>
<html lang="en">
<head>
    <title>Registration Successful</title>
</head>
<body>
    <h1>Registration Successful</h1>
    <p>Email: {{ email }}</p>
    <p>Secret Key: {{ secret }}</p>
    <p>Scan the QR code below with your authenticator app:</p>
    <img src="data:image/png;base64,{{ qr_code }}" alt="QR Code">
    <a href="{{ url_for('index') }}">Home</a>
</body>
</html>


In [ ]:
<!doctype html>
<html lang="en">
<head>
    <title>Verify OTP</title>
</head>
<body>
    <h1>Verify OTP</h1>
    {% with messages = get_flashed_messages(with_categories=true) %}
      {% if messages %}
        <ul>
          {% for category, message in messages %}
            <li class="{{ category }}">{{ message }}</li>
          {% endfor %}
        </ul>
      {% endif %}
    {% endwith %}
    <form method="post">
        <label for="email">Email:</label>
        <input type="email" name="email" required><br>
        <label for="otp">OTP:</label>
        <input type="text" name="otp" required><br>
        <button type="submit">Verify</button>
    </form>
    <a href="{{ url_for('index') }}">Home</a>
</body>
</html>
